# 04 -- Full Optimizer Comparison Report

A consolidated view across every metric (training loss, validation loss, test
accuracy, gradient norm, convergence speed, runtime, iterations, stability,
peak memory) and every task (a9a logistic regression, MNIST softmax
regression), using the pre-generated results in `../results/*.csv`
(produced by `python -m experiments.generate_results`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

RESULTS = Path.cwd().parent / 'results'

a9a_table = pd.read_csv(RESULTS / 'a9a_comparison.csv', index_col='optimizer')
mnist_table = pd.read_csv(RESULTS / 'mnist_comparison.csv', index_col='optimizer')

a9a_table['dataset'] = 'a9a'
mnist_table['dataset'] = 'mnist'

combined = pd.concat([a9a_table, mnist_table])
combined

## Ranking optimizers by test accuracy

In [ ]:
combined.sort_values(['dataset', 'test_accuracy'], ascending=[True, False])[
    ['dataset', 'test_accuracy', 'final_train_loss', 'convergence_epoch', 'runtime_sec', 'peak_memory_mb', 'stability']
]

## Ranking optimizers by convergence speed (epochs to near-best loss)

In [ ]:
combined.sort_values(['dataset', 'convergence_epoch'])[['dataset', 'convergence_epoch', 'test_accuracy', 'runtime_sec']]

## Takeaways

- **Adam** and **RMSProp** consistently reach a good loss within the fewest epochs on both
  tasks, reflecting their per-coordinate adaptive step sizes.
- **L-BFGS** converges in very few *iterations* (it's a quasi-Newton method) but each
  iteration is more expensive (full-batch gradient + line search), so wall-clock
  runtime is the fairer comparison -- see `plot_runtime_vs_loss` in notebooks 02/03.
- **Plain SGD/GD** need many more epochs and careful learning-rate tuning to match
  the adaptive methods, but have the cheapest per-step cost and lowest memory footprint.
- **Adagrad**'s monotonically-growing accumulator causes its effective step size to
  decay over training -- visible as a plateauing loss curve in notebooks 02/03.

See the project **README** for the full write-up, all figures, and the mathematical
background behind each optimizer.